In [ ]:
# === Setup ===
# Runtime: <3m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: optional
# Competition-safe: No — learning profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
import torch
torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
DEVICE = torch.device("cuda" if RUNTIME_PROFILE == "gpu" else "cpu")
if RUNTIME_PROFILE == "gpu" and not torch.cuda.is_available(): raise RuntimeError("GPU profile requested but CUDA is unavailable")
torch.set_default_device(DEVICE)
_device_probe = (torch.ones(8, device=DEVICE) @ torch.ones(8, device=DEVICE)).item()
print(f"Compute device: {DEVICE}; probe={_device_probe:.1f}")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Reference solution — Image Classification

Data → EDA → CNN → validation → deterministic submission.

In [ ]:
def make_images(n=120,seed=42):
    rng=np.random.default_rng(seed); X=np.zeros((n,1,16,16),np.float32); y=np.arange(n)%2
    for i,label in enumerate(y):
        if label==0: X[i,0,3:13,6:10]=1.0
        else: X[i,0,6:10,3:13]=1.0
        X[i]+=rng.normal(0,.08,X[i].shape)
    order=rng.permutation(n); return X[order],y[order]
X,y=make_images(80 if FAST_MODE else 240); cut=int(.75*len(X)); Xtr,Xva=X[:cut],X[cut:]; ytr,yva=y[:cut],y[cut:]
print("train/val",Xtr.shape,Xva.shape,"balance",np.bincount(ytr))

In [ ]:
from torch.utils.data import TensorDataset,DataLoader
device=torch.device("cuda" if torch.cuda.is_available() and not FAST_MODE else "cpu")
train=TensorDataset(torch.from_numpy(Xtr),torch.from_numpy(ytr).long()); loader=DataLoader(train,batch_size=16,shuffle=False)
model=torch.nn.Sequential(torch.nn.Conv2d(1,8,3,padding=1),torch.nn.ReLU(),torch.nn.AdaptiveAvgPool2d(4),torch.nn.Flatten(),torch.nn.Linear(8*4*4,2)).to(device)
opt=torch.optim.Adam(model.parameters(),lr=.01); loss_fn=torch.nn.CrossEntropyLoss()
for _ in range(3 if FAST_MODE else 15):
    model.train()
    for xb,yb in loader: opt.zero_grad(); loss=loss_fn(model(xb.to(device)),yb.to(device)); loss.backward(); opt.step()
model.eval()
with torch.no_grad(): pred=model(torch.from_numpy(Xva).to(device)).argmax(1).cpu().numpy()
accuracy=(pred==yva).mean(); assert accuracy>.9
submission=np.c_[np.arange(len(pred)),pred]; assert submission.shape==(len(yva),2)
print("validation accuracy",accuracy,"submission head",submission[:5])